# Tesing

In [9]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit

# 1. ĐỌC VÀ CHUẨN BỊ (Sửa lỗi month và fillna)
df_sales = pd.read_csv('../dataset/sales.csv', parse_dates=['Date'])
df_sample = pd.read_csv('../dataset/sample_submission.csv', parse_dates=['Date'])
df_sales['month'] = df_sales['Date'].dt.month

def build_ultimate_features(df, source_df):
    df = df.copy()
    df['month'] = df['Date'].dt.month
    df['year'] = df['Date'].dt.year
    
    # --- Key 1: Cyclical Month Features ---
    df['sin_month'] = np.sin(2 * np.pi * df['month']/12)
    df['cos_month'] = np.cos(2 * np.pi * df['month']/12)
    
    # --- Key 2: Historical Statistics (Target Encoding) ---
    month_stats = source_df.groupby('month')['Revenue'].agg(['mean', 'std']).to_dict()
    df['month_avg_rev'] = df['month'].map(month_stats['mean'])
    df['month_std_rev'] = df['month'].map(month_stats['std'])
    
    # --- Key 3: Growth Momentum (Lag 12m) ---
    if 'Revenue' in df.columns:
        df['lag_12m'] = df['Revenue'].shift(12)
    else:
        last_year_vals = source_df['Revenue'].tail(12).values
        df['lag_12m'] = np.resize(last_year_vals, len(df))
        
    return df.bfill().ffill()

train_df = build_ultimate_features(df_sales, df_sales)
test_df = build_ultimate_features(df_sample, df_sales)

features = ['month', 'year', 'sin_month', 'cos_month', 'month_avg_rev', 'month_std_rev', 'lag_12m']

# 2. HUẤN LUYỆN K-FOLD VỚI TUNING SÂU
tscv = TimeSeriesSplit(n_splits=5)

def train_final(X, y, test_X):
    # Log transformation để thu hẹp khoảng cách sai số (Tối ưu MAPE/MAE)
    y_log = np.log1p(y)
    fold_preds = []
    
    for train_idx, val_idx in tscv.split(X):
        X_t, X_v = X.iloc[train_idx], X.iloc[val_idx]
        y_t, y_v = y_log.iloc[train_idx], y_log.iloc[val_idx]
        
        model = xgb.XGBRegressor(
            n_estimators=2000,      # Tăng số lượng cây
            learning_rate=0.005,    # Học cực chậm để cực chính xác
            max_depth=8,            # Tăng độ sâu để bắt pattern phức tạp
            subsample=0.6,          # Random hóa dữ liệu để tránh học vẹt
            colsample_bytree=0.6,
            early_stopping_rounds=100,
            objective='reg:squarederror'
        )
        
        model.fit(X_t, y_t, eval_set=[(X_v, y_v)], verbose=False)
        fold_preds.append(np.expm1(model.predict(test_X)))
        
    return np.mean(fold_preds, axis=0)

# 3. CHẠY VÀ XUẤT FILE
print("Đang huấn luyện Ultimate Model cho Revenue & COGS...")
df_sample['Predicted_Revenue'] = train_final(train_df[features], train_df['Revenue'], test_df[features])
df_sample['Predicted_COGS'] = train_final(train_df[features], train_df['COGS'], test_df[features])

# Lưu file định dạng chuẩn 2 chữ số thập phân
df_sample[['Date', 'Predicted_Revenue', 'Predicted_COGS']].to_csv('submission.csv', index=False, float_format='%.2f')
print("--- ĐÃ XUẤT FILE SUBMISSION MỚI ---")

Đang huấn luyện Ultimate Model cho Revenue & COGS...
--- ĐÃ XUẤT FILE SUBMISSION MỚI ---
